# 01 - Camada Bronze
Criação do banco de dados e tabelas a partir de cada CSV, faz a leitura dos dados sem qualquer alteração na estrutura ou no conteúdo, além de adicionar o exato momento de inserção do dado na camada Bronze. Grava todas as tabelas em formato Delta, utilizando o modo Append

In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

base = "/Volumes/workspace/movies/dados_movies/"

arquivos = {
    "movies_info_IMDB_TMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews",
}

for arq, tabela in arquivos.items():
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("escape", '"')
          .csv(base + arq))
    (df.withColumn("ingestion_datetime", F.current_timestamp())
       .write.format("delta").mode("append").saveAsTable(tabela))
    print(f"OK: {tabela}")

OK: bronze.tb_movies_info
OK: bronze.tb_movies_financials
OK: bronze.tb_movies_metrics
OK: bronze.tb_credits_and_tags
OK: bronze.tb_movies_reviews


In [0]:
import requests
from datetime import date, timedelta

# Widgets: o usuário pode informar as datas no formato MM-DD-AAAA.
# Se ficarem vazios, usa os últimos 7 dias corridos a partir da data de execução
# Isso ocorre pois a API não conta fins de semana e feriados
dbutils.widgets.text("data_inicio", "")
dbutils.widgets.text("data_fim", "")

data_fim = dbutils.widgets.get("data_fim") or f"{date.today():%m-%d-%Y}"
data_inicio = dbutils.widgets.get("data_inicio") or f"{date.today() - timedelta(days=7):%m-%d-%Y}"

url = ("https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
       "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)")

params = {
    "@dataInicial": f"'{data_inicio}'",      # formato exigido pela API: MM-DD-AAAA
    "@dataFinalCotacao": f"'{data_fim}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json",
}

resp = requests.get(url, params=params, timeout=60)
resp.raise_for_status()   # falha cedo se a API retornar erro

In [0]:
from pyspark.sql import functions as F

dados = resp.json()["value"]
if not dados:
    raise ValueError("A API não retornou cotações para o período.")

'''Bronze: sem alterar conteúdo, tudo como string'''
df = spark.createDataFrame(
    [(d["dataHoraCotacao"], str(d["cotacaoCompra"])) for d in dados],
    ["dataHoraCotacao", "cotacaoCompra"],
)

(df.withColumn("ingestion_datetime", F.current_timestamp())
   .write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar"))

display(spark.sql("SELECT * FROM bronze.tb_cotacao_dolar"))

dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-30 13:11:44.783262,5.1803,2026-10-07T17:33:21.301Z
2026-10-01 13:10:35.4469,5.2073,2026-10-07T17:33:21.301Z
2026-10-02 13:03:16.256632,5.2232,2026-10-07T17:33:21.301Z
2026-10-05 13:07:36.558602,4.9853,2026-10-07T17:33:21.301Z
2026-10-06 13:03:21.267287,4.9692,2026-10-07T17:33:21.301Z
2026-10-07 13:05:23.249063,4.9929,2026-10-07T17:33:21.301Z
2026-09-30 13:11:44.783262,5.1803,2026-10-15T18:48:34.222Z
2026-10-01 13:10:35.4469,5.2073,2026-10-15T18:48:34.222Z
2026-10-02 13:03:16.256632,5.2232,2026-10-15T18:48:34.222Z
2026-10-05 13:07:36.558602,4.9853,2026-10-15T18:48:34.222Z
